In [ ]:
from flask import Flask, jsonify, request
import yfinance as yf
import threading

# Create Flask app
app = Flask(__name__)

# -------------------------------
# Company Info Endpoint
# -------------------------------
@app.route("/api/company/<symbol>", methods=["GET"])
def company_info(symbol):
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info

        # Get key officers if available
        officers = info.get("companyOfficers", [])
        key_officers = [{"name": o.get("name"), "title": o.get("title")} for o in officers]

        data = {
            "symbol": symbol.upper(),
            "full_name": info.get("longName", "N/A"),
            "business_summary": info.get("longBusinessSummary", "N/A"),
            "industry": info.get("industry", "N/A"),
            "sector": info.get("sector", "N/A"),
            "key_officers": key_officers
        }
        return jsonify(data), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


    

# -------------------------------
# Function to run Flask in background
# -------------------------------
def run_app():
    app.run(port=5001, debug=True, use_reloader=False)

# Start Flask app in a new thread
thread = threading.Thread(target=run_app)
thread.start()


# http://127.0.0.1:5001/api/company/AAPL



 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Mar/2026 22:37:51] "GET /api/company/AAPL HTTP/1.1" 200 -


In [ ]:
from flask import Flask, jsonify, request
import yfinance as yf
import threading


# Create Flask app
app = Flask(__name__)

# -------------------------------
# Real-Time Stock Data Endpoint
# -------------------------------
@app.route("/api/stock/<symbol>/quote", methods=["GET"])
def stock_quote(symbol):
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info

        previous_close = info.get("previousClose", None)
        current_price = info.get("currentPrice", None)

        if previous_close and current_price:
            change = current_price - previous_close
            percent_change = (change / previous_close) * 100
        else:
            change = percent_change = None

        data = {
            "symbol": symbol.upper(),
            "current_price": current_price,
            "price_change": change,
            "percent_change": percent_change,
            "day_low": info.get("dayLow"),
            "day_high": info.get("dayHigh"),
            "volume": info.get("volume"),
            "market_cap": info.get("marketCap"),
            "open": info.get("open"),
            "previous_close": previous_close
        }
        return jsonify(data), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


def run_app():
    app.run(port=5002, debug=True, use_reloader=False)

# Start Flask app in a new thread
thread = threading.Thread(target=run_app)
thread.start()

# http://127.0.0.1:5002/api/stock/AAPL/quote

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Mar/2026 22:38:15] "GET /api/stock/AAPL/quote HTTP/1.1" 200 -
127.0.0.1 - - [03/Mar/2026 22:38:31] "GET /api/stock/history HTTP/1.1" 404 -


In [ ]:
from flask import Flask, jsonify, request
import yfinance as yf
import threading
import requests

# Create Flask app
app = Flask(__name__)


# -------------------------------
# Historical Data Endpoint
# -------------------------------
@app.route("/api/stock/history", methods=["POST"])
def historical_data():
    try:
        payload = request.get_json()
        symbol = payload.get("symbol")
        start_date = payload.get("start_date")
        end_date = payload.get("end_date")
        interval = payload.get("interval", "1d")  # default daily

        if not symbol or not start_date or not end_date:
            return jsonify({"error": "symbol, start_date, and end_date are required"}), 400

        ticker = yf.Ticker(symbol)
        df = ticker.history(start=start_date, end=end_date, interval=interval)

        # Convert DataFrame to JSON
        df = df.reset_index()
        df["Date"] = df["Date"].astype(str)  # convert dates to string
        data = df.to_dict(orient="records")

        return jsonify({"symbol": symbol.upper(), "historical_data": data}), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500
    
    

def run_app():
    app.run(port=5003, debug=True, use_reloader=False)

# Start Flask app in a new thread
thread = threading.Thread(target=run_app)
thread.start()



payload = {
    "symbol": "AAPL",
    "start_date": "2023-01-01",
    "end_date": "2023-03-01",
    "interval": "1d"
}

response = requests.post("http://127.0.0.1:5000/api/stock/history", json=payload)
data = response.json()

# Print first 5 rows
print(data["historical_data"][:5])
# http://127.0.0.1:5003/api/stock/history

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Mar/2026 22:43:57] "POST /api/stock/history HTTP/1.1" 200 -


[{'Close': 123.09601593017578, 'Date': '2023-01-03 00:00:00-05:00', 'Dividends': 0.0, 'High': 128.83399514878522, 'Low': 122.21021915687207, 'Open': 128.2237854341601, 'Stock Splits': 0.0, 'Volume': 112117500}, {'Close': 124.36566162109375, 'Date': '2023-01-04 00:00:00-05:00', 'Dividends': 0.0, 'High': 126.62936374107429, 'Low': 123.10586505340991, 'Open': 124.88729543343348, 'Stock Splits': 0.0, 'Volume': 89113600}, {'Close': 123.04681396484375, 'Date': '2023-01-05 00:00:00-05:00', 'Dividends': 0.0, 'High': 125.75341088924934, 'Low': 122.79092293482029, 'Open': 125.12351256937882, 'Stock Splits': 0.0, 'Volume': 80962700}, {'Close': 127.57420349121094, 'Date': '2023-01-06 00:00:00-05:00', 'Dividends': 0.0, 'High': 128.23362708296847, 'Low': 122.91886125861728, 'Open': 124.02118700839063, 'Stock Splits': 0.0, 'Volume': 87754700}, {'Close': 128.0958251953125, 'Date': '2023-01-09 00:00:00-05:00', 'Dividends': 0.0, 'High': 131.30438194257326, 'Low': 127.83993420444634, 'Open': 128.41078181

In [3]:
import yfinance as yf

# 1️⃣ Create a Ticker object
ticker = yf.Ticker("AAPL")  # Replace "AAPL" with any stock symbol

# 2️⃣ Get the info dictionary
info = ticker.info

# 3️⃣ Inspect all available keys
print(info.keys())

dict_keys(['address1', 'city', 'state', 'zip', 'country', 'phone', 'website', 'industry', 'industryKey', 'industryDisp', 'sector', 'sectorKey', 'sectorDisp', 'longBusinessSummary', 'fullTimeEmployees', 'companyOfficers', 'auditRisk', 'boardRisk', 'compensationRisk', 'shareHolderRightsRisk', 'overallRisk', 'governanceEpochDate', 'compensationAsOfEpochDate', 'irWebsite', 'executiveTeam', 'maxAge', 'priceHint', 'previousClose', 'open', 'dayLow', 'dayHigh', 'regularMarketPreviousClose', 'regularMarketOpen', 'regularMarketDayLow', 'regularMarketDayHigh', 'dividendRate', 'dividendYield', 'exDividendDate', 'payoutRatio', 'fiveYearAvgDividendYield', 'beta', 'trailingPE', 'forwardPE', 'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 'bid', 'ask', 'bidSize', 'askSize', 'marketCap', 'nonDilutedMarketCap', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh', 'allTimeHigh', 'allTimeLow', 'priceToSalesTrailing12Months', 'fiftyDayAverage', 'twoHundredDayAverage', 

¡Claro que sí! Aquí tienes algunas opciones amigables:

1.  **Hola** - (Hello) - Simple, universal, and always good.
2.  **¡Hola! ¿Cómo estás?** - (Hello! How are you?) - Very common and friendly.
3.  **¡Hola! ¿Qué tal?** - (Hello! What's up?/How's it going?) - A bit more casual but very friendly.
4.  **Buenos días / Buenas tardes / Buenas noches** - (Good morning / Good afternoon / Good evening) - Polite and depends on the time of day.

You can't go wrong with just **"Hola"** or **"¡Hola! ¿Cómo estás?"**


In [ ]:
from flask import Flask, jsonify, request
import yfinance as yf
import threading
from getpass import getpass
from google import genai

# -------------------------------
# Ask for Gemini API Key at startup
# -------------------------------
gemini_key = getpass("Enter your Gemini API Key: ")
client = genai.Client(api_key=gemini_key)

# -------------------------------
# Create Flask app
# -------------------------------
app = Flask(__name__)

# -------------------------------
# Company & Stock Info Endpoint
# -------------------------------
@app.route("/api/company/<symbol>", methods=["GET"])
def company_info(symbol):
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info

        # Company info variables
        company_name = info.get("longName", "N/A")
        industry = info.get("industry", "N/A")
        sector = info.get("sector", "N/A")
        summary = info.get("longBusinessSummary", "N/A")
        website = info.get("website", "N/A")
        full_time_employees = info.get("fullTimeEmployees", None)

        officers = info.get("companyOfficers", [])
        key_officers = [{"name": o.get("name"), "title": o.get("title")} for o in officers]

        # Real-time stock data variables
        current_price = info.get("currentPrice", None)
        previous_close = info.get("previousClose", None)
        day_low = info.get("dayLow", None)
        day_high = info.get("dayHigh", None)
        volume = info.get("volume", None)
        market_cap = info.get("marketCap", None)

        if current_price and previous_close:
            price_change = current_price - previous_close
            percent_change = (price_change / previous_close) * 100
        else:
            price_change = percent_change = None

        # Historical data
        historical_df = ticker.history(period="1mo")  # last 1 month
        historical_data = historical_df.reset_index().to_dict(orient="records")

        # Gemini LLM summary
        llm_prompt = f"""Perform a comprehensive analysis for {company_name} and deliver actionable insightsusing the following data: industry={industry}, sector={sector}, full_time_employees={full_time_employees}
        current_price={current_price}, previous_close={previous_close}, day_low={day_low}, day_high={day_high}, volume={volume}, market_cap={market_cap}
        price_change={price_change}, percent_change={percent_change}, historical_data={historical_data}
        """
        llm_response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=llm_prompt
        )
        company_summary_llm = llm_response.text

        # Return all variables in JSON
        data = {
            "symbol": symbol.upper(),
            "company_name": company_name,
            "industry": industry,
            "sector": sector,
            "website": website,
            "full_time_employees": full_time_employees,
            "key_officers": key_officers,
            "current_price": current_price,
            "previous_close": previous_close,
            "price_change": price_change,
            "percent_change": percent_change,
            "day_low": day_low,
            "day_high": day_high,
            "volume": volume,
            "market_cap": market_cap,
            "long_summary": summary,
            "llm_summary": company_summary_llm,
            "historical_data": historical_data
        }

        return jsonify(data), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


# -------------------------------
# Run Flask in background
# -------------------------------
def run_app():
    app.run(port=5004, debug=True, use_reloader=False)


thread = threading.Thread(target=run_app)
thread.start()

# Now you can access: http://127.0.0.1:5004/api/company/AAPL

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Mar/2026 23:46:56] "GET /api/company/AAPL HTTP/1.1" 200 -
127.0.0.1 - - [03/Mar/2026 23:47:22] "GET /api/company/AAPL HTTP/1.1" 200 -
